# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.** The lane is refresh-opportunity scoring, and the decision it serves is
which visible pages an editor reviews first. That's an ordering problem, not a yes/no one.
I could put a *will this page decline?* classifier underneath, but what the editor actually
consumes is a sorted queue cut at the top K they have time for. So the task type that matches
the decision is ranking: a priority score per page, evaluated at the top of the list.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
visible = df[df["impressions_90d"] >= 100].copy()

print(f"all pages: {len(df):,}    visible (impressions_90d >= 100): {len(visible):,}")
print("task: order these visible pages by refresh priority, then cut at the top K.")


all pages: 30,000    visible (impressions_90d >= 100): 22,006
task: order these visible pages by refresh priority, then cut at the top K.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I'd want to predict is an **observed future drop**: take a page's state now, look at the
next 30 days, label it 1 if impressions or sessions fell past a threshold. That label is
measured, not defined, which is what keeps the model learning the world instead of someone's
rule. The starter slice doesn't carry a forward window, though. Its only decline signal is
`is_declining_label`, and per the data notes that's derived from `trend_direction` / `trend_pct`,
so it's a **defined proxy, not an outcome**. I use it below only to make the framing concrete
and to check a metric is computable today. The real target needs the warehouse daily facts
(prior 90d → next 30d). `trend_pct`, `trend_direction` and `is_declining_label` never become
features — that's the leakage trap.

In [2]:
proxy = df["is_declining_label"]
print("proxy base rate:", round(proxy.mean(), 3), "of", f"{len(df):,}", "pages")
print("never-features (leakage):", ["trend_pct", "trend_direction", "is_declining_label"])

sketch = df[["content_id", "impressions_prev_30d", "impressions_last_30d"]].head(5).copy()
sketch["decline_label (illustrative)"] = (
    sketch["impressions_last_30d"] < 0.8 * sketch["impressions_prev_30d"]
).astype(int)
sketch


proxy base rate: 0.542 of 30,000 pages
never-features (leakage): ['trend_pct', 'trend_direction', 'is_declining_label']


,content_id,impressions_prev_30d,impressions_last_30d,decline_label (illustrative)
0,content_304f48230142,987,578,1
1,content_a1fb4e703a9e,5915,2501,1
2,content_9aa793d4d895,6089,2382,1
3,content_331d6c4de07b,4206,3626,0
4,content_d99b7a2d90ca,6452,4211,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K.** An editor works a short list, so what matters is how many of the top K flagged
pages genuinely needed attention, not global accuracy over 30,000 rows. I'll report P@20 and
P@50 against a held-out label, with the base rate next to it so the number has a reference.
Below I compute it on a plain baseline to show it's a number I can produce now, not a promise
for later.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining_label"].values
stale = (df["days_since_last_update"] >= 180).astype(int)
vis = (df["impressions_90d"] >= 500).astype(int)
baseline_score = stale * vis * df["impressions_90d"]

for k in (20, 50):
    p = precision_at_k(baseline_score, y, k)
    print(f"baseline Precision@{k}: {p:.3f}    base rate {y.mean():.3f}")


baseline Precision@20: 0.900    base rate 0.542
baseline Precision@50: 0.680    base rate 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page**, over its trailing-90-day window, keyed by the pseudonymous
`content_id`. The lane's slice is the *visible* pages — enough impressions to be worth a look —
since scoring a page nobody sees isn't a refresh decision. Here's that slice as a real dataframe.
`ctr` and `engagement_rate` are ×100 percentages, and `avg_position == 0` means no data, not rank 0.

In [4]:
cols = ["content_id", "impressions_90d", "avg_position", "ctr",
        "days_since_last_update", "content_age_days", "word_count", "engagement_rate"]
lane_slice = visible[cols].reset_index(drop=True)

print(f"one row = one visible content page. rows: {len(lane_slice):,}")
lane_slice.head(8)


one row = one visible content page. rows: 22,006


,content_id,impressions_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,engagement_rate
0,content_304f48230142,3803,10.6,0.76,20,187,3221.0,5.88
1,content_a1fb4e703a9e,15320,20.3,0.05,25,445,2481.0,0.00
2,content_9aa793d4d895,12581,36.5,0.09,20,141,3515.0,0.00
3,content_331d6c4de07b,11751,6.2,0.49,22,463,NaN,1.28
4,content_d99b7a2d90ca,19140,44.0,0.13,14,263,2803.0,0.00
5,content_d4084a4bc775,3970,8.5,0.03,20,147,3080.0,0.00
6,content_a63219c6e95a,1724,21.2,0.06,22,445,NaN,3.57
7,content_5e6c160719bc,32574,46.0,0.09,20,90,3807.0,5.88


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A one-line rule like *stale and visible* gets you a decent first pass, and it's readable, which
is exactly why I'd ship it as the baseline. Where it runs out is the interactions: whether a
stale page is worth refreshing depends on position, CTR, impression volume, age and engagement
*together*, and those signals are weak on their own and tangled with each other. Missingness
itself tracks `content_type`, so a threshold on one column quietly encodes another. In notebook
02 a small tree combining these out-ranked the hand rule on held-out pages while staying
readable. That's the narrow, earned case for ML here — the pattern is real but too multivariate
to hand-write — and the model still has to beat the rule on holdout before it's worth the
complexity. The single-signal correlations below show why no one threshold does the job.

In [5]:
features = ["impressions_90d", "avg_position", "ctr", "days_since_last_update",
            "content_age_days", "word_count", "engagement_rate"]

d = df.copy()
d.loc[d["avg_position"] == 0, "avg_position"] = np.nan
corr = (d[features].corrwith(df["is_declining_label"])
        .abs().sort_values(ascending=False).round(3))

print("|correlation| of each single feature with the proxy label:")
print(corr.to_string())


|correlation| of each single feature with the proxy label:
content_age_days          0.164
word_count                0.090
days_since_last_update    0.081
avg_position              0.081
ctr                       0.062
impressions_90d           0.018
engagement_rate           0.013


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.